In [ ]:
###########   PHASE 1  #################
######STEP3- Urbanisation analysis 

##Purpose
#This notebook investigates changes in population growth and urban development in Lagos and Durban. 
#The objective is to evaluate how urban expansion contributes to increasing flood exposure.

##Research Context
#Urbanisation alters natural drainage systems through land cover change, 
#increased impervious surfaces and population concentration. 
#These changes often amplify flood risk independently of climatic trends.

##Input Data
#Population datasets
#Urbanisation indicators
#Built surface information

##Methods
#Descriptive statistics
#Growth trend analysis
#Time series visualisation
#Comparative city analysis

##Outputs
#Population growth trends
#Urbanisation trend plots
#Comparative statistics




#Step 3a-Load World Bank CSV files 
#World Bank CSVs have a peculiar structure:
#Row 1-4 are metadata headers (not data)
# -> The actual data starts at row 5
# -> Years are stored as COLUMNS not rows
#This is "wide format" — opposite of what we want for plotting
#We need to reshape it to "long format" (one row per country per year)
#In R this is pivot_longer(). In Python it is pd.melt()

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")

# Use ONE consistent output directory (do not overwrite later)
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project"
#Load urban population
pop_raw = pd.read_csv(
    os.path.join(BASE_DIR,
                 "data/raw/urban/urban_population/world_bank_urban_population_total.csv"),
    skiprows=4   ##Skip the 4 World Bank metadata rows at the top
)

#Load urban land area 
land_raw = pd.read_csv(
    os.path.join(BASE_DIR,
                 "data/raw/urban/urban_land_area/world_bank_urban_land_area_km2.csv"),
    skiprows=4
)

#urban population growth rate
growth_raw = pd.read_csv(
    os.path.join(BASE_DIR,
                 "data/raw/urban/urban_population_growth/world_bank_urban_population_growth_pct.csv"),
    skiprows=4
)



In [ ]:
#STEP 3b - Filter to Nigeria and South Africa, reshape to long format
#Define countries of interest
#World Bank uses these exact country names

COUNTRIES = {
    "Nigeria": "Lagos",         #We use national data as city-level proxy
    "South Africa": "Durban"      # and label it by city for consistency
}


#Reusable function to clean any World Bank dataset
#Writing a function avoids repeating the same code 3 times
#In R I would write a function the same way: my_fn <- function(df, ...) { ... }

def clean_wb(df, value_name):

    df = df[df["Country Name"].isin(COUNTRIES.keys())].copy()

    year_cols = [c for c in df.columns if c.isdigit()]

    df = df[["Country Name"] + year_cols]

    long = pd.melt(
        df,
        id_vars=["Country Name"],
        value_vars=year_cols,
        var_name="Year",
        value_name=value_name
    )

    long["Year"] = long["Year"].astype(int)
    long["City"] = long["Country Name"].map(COUNTRIES)

    return long.dropna()
    
#Apply function to all three datasets
pop_long = clean_wb(pop_raw, "Urban_population")
land_long = clean_wb(land_raw, "Urban_land_km2")
growth_long = clean_wb(growth_raw, "Urban_growth_pct")


#Quick check
print("\n--- Urban population (long format, first 5 rows) ---")
print(pop_long.head())
print(f"\nShape: {pop_long.shape}")
print(f"Years covered: {pop_long['Year'].min()} – {pop_long['Year'].max()}")
print(f"Cities: {pop_long['City'].unique()}")

In [ ]:
#STEP 3c - Urbanisation trends (PLOTTING)
#What it does:
#Shows how urban population, land expansion, and growth rate change over time.

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
#fig, axes creates 3 side-by-side plots
#axes[0], axes[1], axes[2] are like three separate ggplot objects
#In R: cowplot::plot_grid() or patchwork

colors = {"Lagos": "#2E86C1", "Durban": "#1E8449"}

#plot 1: Urban Population
for city in pop_long["City"].unique():
    d = pop_long[pop_long["City"] == city]
    axes[0].plot(d["Year"], d["Urban_population"]/1e6, label=city, color=colors[city])
axes[0].set_title("Urban population (millions)")

#plot 2: Urban land area
for city in land_long["City"].unique():
    d = land_long[land_long["City"] == city]
    axes[1].plot(d["Year"], d["Urban_land_km2"], label=city, color=colors[city])
axes[1].set_title("Urban land area (km²)")

#plot 3: Urban population growth rate
for city in growth_long["City"].unique():
    d = growth_long[growth_long["City"] == city]
    axes[2].plot(d["Year"], d["Urban_growth_pct"], label=city, color=colors[city])
axes[2].axhline(0, linestyle="--", color="grey", linewidth=1)
axes[2].set_title("Urban growth rate (%)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "urban_trends.png"), dpi=150)
plt.show()

In [ ]:
#STEP 3d - Decade summaries + flood riak indicator
#What it does:
#Compresses time series into decades and builds a density proxy.
#Add decade column
pop_long["Decade"] = (pop_long["Year"] // 10) * 10
land_long["Decade"] = (land_long["Year"] // 10) * 10
#Decade means
pop_dec = pop_long.groupby(["Decade", "City"])["Urban_population"].mean()/1e6  #convert to millions
pop_dec = pop_dec.reset_index(name="Pop_millions")

land_dec = land_long.groupby(["Decade", "City"])["Urban_land_km2"].mean().reset_index()

#Merge population and land into one table ---
#merge() = merge() or left_join() in R
urban = pop_dec.merge(land_dec, on=["Decade", "City"])

#Compute population density as a flood-risk proxy
#Higher urban density -> more impervious surface per km²
# -> less natural drainage -> higher runoff = higher flood risk
urban["Density"] = (urban["Pop_millions"]*1e6) / urban["Urban_land_km2"]

print(urban)

In [ ]:
#STEP 3e - Percentage growth (headline numbers)
#What it does:
#Shows how much cities expanded over full period.
for city in pop_long["City"].unique():

    d = pop_long[pop_long["City"] == city]

    start = d.loc[d["Year"] == d["Year"].min(), "Urban_population"].values[0]
    end = d.loc[d["Year"] == d["Year"].max(), "Urban_population"].values[0]

    growth = ((end - start) / start) * 100 #Percentage growth — same formula as in R

    print(city)
    print("Start:", start)
    print("End:", end)
    print("Growth %:", round(growth, 1))
    print()

In [ ]:
!pip install rasterio

In [ ]:
import rasterio                          #reads .tif raster files
import numpy as np
# rasterio -> terra / raster package in R
import os

LULC_DIR = r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw"

for root, dirs, files in os.walk(LULC_DIR):
    for file in files:
        if file.endswith(".tif"):
            print(os.path.join(root, file))

In [ ]:
#STEP 3f - GHSL raster analysis -built-up surface
#What it does:
#Measures actual built-up surface (satellite-based urban footprint).
import rasterio

rasters = {
    "Lagos": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\lulc\ghs_built_surface_2020_durban.tif",

    "Durban": r"C:\Users\drsih\Desktop\profile\flooding_project\data\raw\lulc\ghs_built_surface_2020_lagos.tif"
}


raster_stats = {}

for city, path in rasters.items():

    with rasterio.open(path) as src:
        #Read all pixel values into a numpy array
        #In R: terra::values(rast(path))
        data = src.read(1).astype(float) #band 1 = built-up surface in m²
        
         #Replace nodata values with NaN
        if src.nodata is not None:
            data[data == src.nodata] = np.nan

        valid = data[~np.isnan(data)]

        raster_stats[city] = {
            "Total built-up": np.nansum(valid),
            "Mean intensity": np.nanmean(valid[valid > 0]),
            "Built-up %": np.sum(valid > 0) / len(valid) * 100
        }

print(pd.DataFrame(raster_stats).T)

In [ ]:
#STEP 3g - Raster visualisation as simple maps
#What it does:
#Maps built-up intensity spatially.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, (city, path) in enumerate(rasters.items()):

    with rasterio.open(path) as src:
        data = src.read(1).astype(float)

        if src.nodata is not None:
            data[data == src.nodata] = np.nan

    axes[i].imshow(data, cmap="YlOrRd")
    axes[i].set_title(city)
plt.suptitle("Built-up surface density: Lagos and Durban (GHSL 2020, 1km resolution)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "ghsl_builtup_maps.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("Raster maps saved.")

In [ ]:
pop_long.to_csv("outputs/urban_population_clean.csv", index=False)
urban.to_csv("outputs/urban_summary.csv", index=False)

In [ ]:
####Summary

##Main Outcomes
#Evaluated long-term urban population growth.
#Compared urbanisation trajectories between Lagos and Durban.
#Identified differences in urban expansion that influence flood exposure.

##Limitations
#National urban indicators were used as proxies for city-level observations where local data were unavailable.

##Next Notebook
#->**04_emdat_disaster_analysis.ipynb**
#The next notebook analyses historical flood disaster records to examine changes in flood frequency, impacts and damages.